# Chapter 7 — Position: Sinusoidal → Learned → RoPE → ALiBi

Every equation, worked example and exercise answer in Chapter 7, recomputed with
`tfs/model.py`'s RoPE (`build_rope_cache`, `rotate_half`, `apply_rope`). Each cell **asserts**
what it checks, so `python tests/run_notebooks.py 07` fails if the chapter and the code disagree.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
from tfs.model import build_rope_cache, rotate_half, apply_rope

## 7.1 / Exercise 7.1 — attention is permutation-equivariant; RoPE and the causal mask break it

In [ ]:
torch.manual_seed(0)
B, T, d = 1, 6, 16
Wq, Wk, Wv = (torch.randn(d, d, dtype=torch.float64) / 4 for _ in range(3))
def attn(x, rope=False, causal=False):
    q, k, v = (x @ Wq)[:, None], (x @ Wk)[:, None], (x @ Wv)[:, None]   # (B, 1, T, d)
    if rope:
        cos, sin = build_rope_cache(x.shape[1], d)
        q, k = apply_rope(q, k, cos.double(), sin.double())
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal)[:, 0]
x = torch.randn(B, T, d, dtype=torch.float64)
perm = torch.randperm(T)
same = lambda kw: torch.allclose(attn(x, **kw)[:, perm], attn(x[:, perm], **kw))
print("no position, no mask:", same({}), "  + RoPE:", same({"rope": True}), "  + causal mask:", same({"causal": True}))
assert same({}) and not same({"rope": True}) and not same({"causal": True})
# and the per-position MLP cannot fix it: it commutes with any permutation
mlp = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, d)).double()
assert torch.allclose(mlp(x)[:, perm], mlp(x[:, perm]))

## 7.2 — sinusoidal encoding: PE(pos + k) is a fixed linear map (a rotation) of PE(pos)

In [ ]:
def sinusoidal(T, d):
    pos = torch.arange(T)[:, None]
    i = torch.arange(0, d, 2)[None, :]
    ang = pos / (10000 ** (i / d))
    pe = torch.zeros(T, d)
    pe[:, 0::2] = torch.sin(ang)
    pe[:, 1::2] = torch.cos(ang)
    return pe.double()
pe = sinusoidal(200, 16)
k = 7
w = 1 / 10000 ** (torch.arange(0, 16, 2).double() / 16)        # per-pair frequency
M = torch.zeros(16, 16, dtype=torch.float64)                    # block-diagonal 2x2 rotations by k*w
for j, a in enumerate(k * w):
    c, s = torch.cos(a), torch.sin(a)
    M[2 * j:2 * j + 2, 2 * j:2 * j + 2] = torch.tensor([[c, -s], [s, c]])
assert torch.allclose(pe[k:], pe[:-k] @ M, atol=1e-5)           # the same M for every position
print("PE[pos+7] == PE[pos] @ M for all", 200 - k, "positions")

## 7.3 — a learned table has a hard ceiling (GPT-2: 1024 rows)

In [ ]:
wpe = nn.Embedding(1024, 768)
assert wpe.weight.numel() == 1024 * 768                           # max_T * d parameters
try:
    wpe(torch.tensor([1024])); raised = False
except IndexError:
    raised = True
assert raised                                                      # position 1025 does not exist
cos, sin = build_rope_cache(40_000, 64)                             # RoPE: any position, no table
assert torch.isfinite(cos).all() and cos.shape == (40_000, 64)

## 7.4 — the 2-D case: Rₘᵀ Rₙ = Rₙ₋ₘ, so only the offset survives

In [ ]:
R = lambda a: torch.tensor([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]], dtype=torch.float64)
for m, n in [(0, 3), (5, 2), (11, 40)]:
    assert torch.allclose(R(m).T @ R(n), R(n - m))
assert torch.allclose(R(0.7).T @ R(0.7), torch.eye(2, dtype=torch.float64))   # orthogonal: length kept

## 7.4 — the worked example (d_head = 8): frequencies, a rotating q, and the cancelling positions

In [ ]:
theta = [10000 ** (-2 * i / 8) for i in range(4)]
for i, t in enumerate(theta):
    print(f"pair i={i}: theta={t:.6f}   full turn every {2 * math.pi / t:,.2f} tokens")
assert [round(t, 6) for t in theta] == [1.0, 0.1, 0.01, 0.001]
assert [round(2 * math.pi / t, 2) for t in theta] == [6.28, 62.83, 628.32, 6283.19]
q1 = torch.tensor([1.0, 0.0], dtype=torch.float64)
rows = [(R(m) @ q1).tolist() for m in range(4)]
print([[round(v, 4) for v in r] for r in rows], [round(math.degrees(m), 1) for m in range(4)])
assert [[round(v, 4) for v in r] for r in rows] == [[1.0, 0.0], [0.5403, 0.8415], [-0.4161, 0.9093], [-0.99, 0.1411]]
assert [round(math.degrees(m), 1) for m in range(4)] == [0.0, 57.3, 114.6, 171.9]
q, k = torch.tensor([0.6, 0.8], dtype=torch.float64), torch.tensor([1.0, 0.0], dtype=torch.float64)
dots = [(R(m) @ q) @ (R(n) @ k) for m, n in [(0, 2), (1, 3), (5, 7), (10, 12)]]
print("offset 2:", [round(v.item(), 6) for v in dots], "  offset 3:", round(((R(1) @ q) @ (R(4) @ k)).item(), 6))
assert all(round(v.item(), 6) == 0.47775 for v in dots)
assert round(((R(1) @ q) @ (R(4) @ k)).item(), 6) == -0.481099

## The chapter's own code cell, in float32

**Resolved (November 2026).** This cell found an error in the chapter, and the chapter text was corrected to match the measurement below. The original finding: The text says the four scores are "bit-identical because the gap was
always 2". Mathematically they are equal; computed in float32 (as the chapter's `rot` does, via
`torch.tensor`) they agree to ~3×10⁻⁸ but are *not* bit-identical. The chapter's own assert
(`< 1e-6`) is the right test; "bit-identical" is not.

In [ ]:
def rot(v, m, theta=1.0):
    a = m * theta
    return torch.tensor([v[0]*math.cos(a) - v[1]*math.sin(a),
                         v[0]*math.sin(a) + v[1]*math.cos(a)])
q, k = torch.tensor([0.6, 0.8]), torch.tensor([1.0, 0.0])
vals = [torch.dot(rot(q, m), rot(k, n)).item() for m, n in [(0, 2), (1, 3), (5, 7), (10, 12)]]
print([f"{v:.10f}" for v in vals])
assert all(round(v, 6) == 0.47775 for v in vals)                  # the printed claim
assert abs(torch.dot(rot(q, 0), rot(k, 2)) - torch.dot(rot(q, 10), rot(k, 12))) < 1e-6
assert len(set(vals)) > 1                                         # ... but not bit-identical

## Exercise 7.2 — the relative-position property for the real implementation (d = 64)

**Minor wording.** The chapter expects the three offset-3 scores to "agree to ~6 decimal places".
In float32 with scores around ±6 they agree to ~6 *significant figures* (differences of ~1e-5);
six decimal places is beyond float32 at that magnitude. The offset-4 score differs in the first digit.

In [ ]:
torch.manual_seed(0)
d = 64
cos, sin = build_rope_cache(200, d)
q, k = torch.randn(1, 1, 1, d), torch.randn(1, 1, 1, d)
def score(m, n):
    qm = q * cos[m] + rotate_half(q) * sin[m]
    kn = k * cos[n] + rotate_half(k) * sin[n]
    return (qm * kn).sum().item()
s = [score(5, 8), score(50, 53), score(100, 103)]
print(s, score(5, 9))
assert max(s) - min(s) < 1e-4 * max(1.0, abs(s[0]))               # agree to ~6 significant digits
assert abs(score(5, 9) - s[0]) > 1e-3                              # offset 4 is different
# apply_rope on (B, nh, T, hd) does the same thing at every position
qq, kk = apply_rope(q.expand(1, 1, 200, d), k.expand(1, 1, 200, d), cos, sin)
assert abs((qq[0, 0, 5] * kk[0, 0, 8]).sum().item() - s[0]) < 1e-5

## 7.4 — two conventions, both correct: half-split (Llama/HF) vs adjacent pairs (the paper)

They differ by a fixed permutation of the feature axis, so one equals the other in a permuted
basis — and they are not interchangeable on the same weights.

In [ ]:
def rope_adjacent(x, pos, base=10000.0):
    d = x.shape[-1]
    inv = 1.0 / base ** (torch.arange(0, d, 2).float() / d)
    a = pos[:, None].float() * inv                               # (T, d/2)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * a.cos() - x2 * a.sin()
    out[..., 1::2] = x1 * a.sin() + x2 * a.cos()
    return out
d, T = 16, 10
x = torch.randn(T, d)
cos, sin = build_rope_cache(T, d)
half = x * cos + rotate_half(x) * sin
P = torch.cat([torch.arange(0, d, 2), torch.arange(1, d, 2)])     # adjacent -> half-split layout
inv = torch.argsort(P)
assert torch.allclose(rope_adjacent(x[:, inv], torch.arange(T))[:, P], half, atol=1e-6)   # equivalent
assert not torch.allclose(rope_adjacent(x, torch.arange(T)), half, atol=1e-3)            # not interchangeable
print("half-split RoPE == adjacent-pair RoPE in the permuted basis; on the same weights they differ")

## 7.5 — ALiBi: a per-head linear distance penalty with geometric slopes

In [ ]:
n_head, T = 8, 6
slopes = torch.tensor([2 ** (-8 * (h + 1) / n_head) for h in range(n_head)])
print("slopes", slopes.tolist())
assert slopes[:3].tolist() == [0.5, 0.25, 0.125]                     # 1/2, 1/4, 1/8, ...
i, j = torch.arange(T)[:, None], torch.arange(T)[None, :]
bias = slopes[:, None, None] * (j - i)                                # (H, T, T)
assert (bias.masked_select((j <= i).expand(n_head, T, T)) <= 0).all()   # a penalty, never a bonus
assert torch.equal(bias[:, 3, 1], bias[:, 5, 3])                      # depends only on j - i

## Exercise 7.3 — the RoPE wavelength spectrum at d_head = 128, base 10,000 vs 500,000

In [ ]:
d = 128
for base, want_long, want_n in ((10_000, 5.4e4, 18), (500_000, 2.6e6, 32)):
    wl = [2 * math.pi * base ** (2 * i / d) for i in range(d // 2)]
    n_long = sum(w > 4096 for w in wl)
    cut = (d / 2) * math.log(4096 / (2 * math.pi)) / math.log(base)     # the closed form
    print(f"base {base:>7,}: shortest {wl[0]:.2f}, longest {wl[-1]:.3g} tokens, "
          f"{n_long}/64 pairs longer than 4096 (closed form: i > {cut:.2f})")
    assert round(wl[0], 2) == 6.28 and round(wl[-1], -int(math.log10(want_long)) + 1) == want_long
    assert n_long == want_n == sum(i > cut for i in range(d // 2))